In [ ]:
import zipfile, io
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DATA = Path("data")


if not (DATA / "sigwatch_data").exists():
    for z in ["ngo_data_bocconi.zip", "banks_data_bocconi.zip"]:
        zipfile.ZipFile(DATA / z).extractall(DATA)


for p in sorted(DATA.rglob("*")):
    print(p)

In [ ]:
# Sigwatch
files = sorted((DATA / "sigwatch_data").glob("*.dta"))

sig = pd.concat(
    [pd.read_stata(f).assign(year_file=int(f.name[:4])) for f in files],
    ignore_index=True,
)
print(sig.shape)

In [ ]:
#Datastream
import sys


ri_sheets = pd.read_excel(DATA / "banks_data_bocconi" / "banks_ri.xlsm", sheet_name=None)
ri_sheets.pop("REQUEST_TABLE")

print(ri_sheets.keys())
print(ri_sheets["US"].shape)

In [ ]:
longs = []

for k, v in ri_sheets.items():
    long = v.melt(id_vars="Name", var_name="bank", value_name="ri")
    long = long.rename(columns={"Name": "date"})
    long["country"] = k
    longs.append(long)

ri = pd.concat(longs, ignore_index=True)

In [ ]:
print(ri.shape)
print(ri["country"].value_counts())
print(ri["bank"].nunique())

## 1. Sigwatch data

In [ ]:
print(f"{sig['uid_archive'].nunique():,} events, {len(sig):,} rows")
print(f"{sig['company'].nunique():,} companies in total") #nbr of unique company_name -> Number of companies
print(f"Average : {len(sig) / sig['uid_archive'].nunique():.1f} companies per event")


**Largest event:**

In [ ]:
top_uid = sig["uid_archive"].value_counts().index[0]
top = sig[sig["uid_archive"] == top_uid]
print(f"Largest event: {len(top)} firms named by {top['ngo_name1'].iloc[0]} on {top['date'].iloc[0]}")

Largest event. The event with the most targets is a single communication by ActionAid UK on 13 May 2013, naming 72 companies at once (BP, Shell, Tesco, AstraZeneca, …).

This matters for the event study: an NGO communication that names dozens of firms is unlikely to be firm-specific news, and any common price move that day would be absorbed by the market factor. We therefore compute n_targets (number of firms named in the same event) and will either drop or analyse separately the mass events above a threshold.

In [ ]:
counts = sig["uid_archive"].value_counts()          #number of firms per event
sig["n_targets"] = sig["uid_archive"].map(counts)   #attach it to every (event, firm) row

In [ ]:
events = sig.drop_duplicates("uid_archive")

print("n_targets, per (event, firm) row:")
print(sig["n_targets"].describe().round(2))

print("\nn_targets, per event:")
print(events["n_targets"].describe().round(2))

share_single = (events["n_targets"] == 1).mean()
print(f"\n{share_single:.1%} of events target a single company")

### **1b. Clean Sigwatch and select bank campaigns**

In [ ]:
#Basic cleaning
import re
sig["date"] = pd.to_datetime(sig["date"])                                    # dates are stored as text
for c in ["isin_1", "isin_2", "isin_3"]:
    sig[c] = sig[c].replace("", np.nan)                                      # empty strings -> missing
sig["parent_country"] = sig["company_parent_country"].replace({"USA": "US"}) # same country, two spellings

In [ ]:
# Names of the banks we have prices for (Datastream, incl. Swiss/Nordic file)

# Read the column headers of every country sheet and strip the " - TOT RETURN IND" suffix to get the bank name
def ds_names(file):
    sheets = pd.read_excel(DATA / "banks_data_bocconi" / file, sheet_name=None, nrows=1)
    sheets.pop("REQUEST_TABLE")
    return {c.rsplit(" - ", 1)[0].strip().lower() for v in sheets.values() for c in v.columns if c != "Name"}

# Combine the main file and the Swiss/Nordic file into one set of priced bank names
ds = ds_names("banks_ri.xlsm") | ds_names("scandinavian_banks_ri.xlsm")

In [ ]:
# igwatch name -> Datastream name, parsed from the Stata file
mp = {}
for line in open(DATA / "example_name_cleaning.txt", encoding="utf-8", errors="ignore"):
    if line.strip().startswith("replace company"):
        q = re.findall(r'"([^"]*)"', line)                      # [target, source1, source2, ...]
        mp.update({s.strip().lower(): q[0].strip().lower() for s in q[1:]})

# remove wrong mappings from the Stata file (e.g. Berkshire Hathaway -> a Swiss cantonal bank)
for k in ["berkshire hathaway", "pinnacle west capital corporation", "virgin group",
          "lloyd's of london", "ace group", "gruppo assicurazioni generali"]:
    mp.pop(k, None)

# fix targets that don't exist in Datastream + add banks the Stata file missed
mp.update({
    "bankinter, s.a.": "bankinter 'r'", "allianz se": "allianz", "allianz group / allianz se": "allianz",
    "jp morgan chase": "jp morgan chase & co.", "goldman sachs group, inc.": "goldman sachs gp.",
    "skandinaviska enskilda banken ab (seb)": "skandinaviska enskilda banken a",
    "handelsbanken": "svenska handelsbanken a", "dnb nor": "dnb", "banco popular espanol": "popular",
})

In [ ]:
# Match on the company, else on its parent (so subsidiaries like Halifax -> Lloyds are kept)
for c in ["company", "company_parent"]:
    s = sig[c].str.strip().str.lower()
    sig[c + "_ds"] = s.map(mp).fillna(s)
sig["ds_name"] = sig["company_ds"].where(sig["company_ds"].isin(ds),
                                         sig["company_parent_ds"].where(sig["company_parent_ds"].isin(ds)))

# Saga Group's Datastream index never changes (no price data), so its events cannot be studied
NO_PRICES = {"saga group"}
banks = sig[sig["ds_name"].notna() & ~sig["ds_name"].isin(NO_PRICES)].copy()

#### **Description of the bank campaigns**

In [ ]:
# Descriptives of the bank campaigns

# Sigwatch gives one reach score per NGO (up to 5): keep the strongest one in the event
banks["ngo_power"] = banks[[f"ngo_power{i}" for i in range(1, 6)]].max(axis=1)

# Size of the sample: rows, events, banks and lead NGOs
print(f"{len(banks):,} rows | {banks['uid_archive'].nunique():,} events | "
      f"{banks['ds_name'].nunique()} banks | {banks['ngo_name1'].nunique():,} lead NGOs")

# Positive vs. negative campaigns
print(banks["sentiment"].map({-2: "very negative", -1: "negative", 0: "neutral", 1: "positive", 2: "very positive"})
      .value_counts().reindex(["very negative", "negative", "neutral", "positive", "very positive"]))

# Distribution of prominence, NGO reach and number of firms named in the same event
print(banks[["prominence", "ngo_power", "n_targets"]].describe().round(2))

# Most active NGOs
print(banks["ngo_name1"].value_counts().head(10))

**Sentiment**: The sample contains 3,408 campaign-bank observations (1,366 distinct events) targeting 72 listed banks, launched by 381 different lead NGOs. Campaigns are overwhelmingly critical: about 80% are negative or very negative (2,712 rows), versus about 15% positive (517) and 5% neutral (179). This is consistent with NGOs mostly using campaigns to pressure banks rather than to praise them, so the event study will have far more negative than positive events.

**Prominence**: Prominence is centred on 2, so a typical campaign mentions the bank in the body of the communication rather than the headline. The lead NGO’s reach is medium, with a quarter of the sample at 2.5 or above (global or international NGOs). n_targets is skewed: the median event names 8 firms but the maximum is 72. Many bank campaigns are therefore rankings or multi-company reports rather than bank-specific attacks. We flag events with more than 10 targets (mass) and treat them as a robustness check.

**Activity**: Activity is concentrated in a few NGOs: Friends of the Earth France alone accounts for 246 rows (about 7% of the sample), followed by Which? (UK, 180) and Rainforest Action Network (US, 120). The top ten are mostly national or regional groups (France, UK, US, Belgium, Netherlands, Spain), so the results may reflect these NGOs’ repeated campaigns rather than the whole NGO universe.

#### **One row per bank-day, then flag problem events**

**One row per bank-day**
A bank can appear several times on the same day (parent and subsidiary, or several NGOs), so we collapse to one row per bank and date. We average sentiment, keep the maximum prominence and NGO reach, and add a `climate` flag from the issue fields.

In [ ]:
#One row per bank-day
issues = banks[["issue_name1", "issue_name2", "issue_name3"]].fillna("").agg(" ".join, axis=1)
banks["climate"] = issues.str.contains("climate|carbon|coal|fossil|oil|gas|pipeline|fracking|emission|renewable", case=False)

events = (banks.sort_values(["ds_name", "date"])
          .groupby(["ds_name", "date"], as_index=False)
          .agg(uid_archive=("uid_archive", "first"), parent_country=("parent_country", "first"),
               isin=("isin_1", "first"), ngo=("ngo_name1", "first"), n_rows=("uid_archive", "size"),
               sentiment=("sentiment", "mean"), min_sentiment=("sentiment", "min"),
               prominence=("prominence", "max"), ngo_power=("ngo_power", "max"),
               n_targets=("n_targets", "min"), climate=("climate", "max")))
events["negative"] = (events["sentiment"] < 0).astype(int)

**Flag problem events and select the final sample**
For the event study we need clean, non-overlapping events. We (1) drop days where the same bank got opposite-sign news within 5 days, since we can't tell which news moved the price, and (2) keep only the first event in each 30-day run, so the [-10, +10] windows don't overlap. Events naming more than 10 firms (`mass`) are only flagged, not dropped.

In [ ]:
# Flag problem events and keep a clean, non-overlapping sample

# Flag bank-days with opposite-sign news within 5 days (can't tell which moved the price)
g = events.groupby("ds_name")
events["days_since_prev"] = g["date"].diff().dt.days
events["opposite"] = (events["days_since_prev"] <= 5) & (events["sentiment"] * g["sentiment"].shift() < 0)
events["opposite"] |= events["opposite"].groupby(events["ds_name"]).shift(-1, fill_value=False)
events["mass"] = events["n_targets"] > 10                    # flag only, not dropped

# Keep an event only if it is >= 30 days after the last kept one
def first_in_window(dates, gap=30):
    keep, last = [], None
    for d in dates:
        ok = last is None or (d - last).days >= gap
        keep.append(ok)
        last = d if ok else last
    return keep

# Apply it bank by bank, after removing the opposite-sign events
ok = events[~events["opposite"]]
events["keep"] = False
events.loc[ok.index, "keep"] = ok.groupby("ds_name")["date"].transform(lambda s: pd.Series(first_in_window(s), index=s.index))
events["keep"] = events["keep"].astype(bool)
print(events[["opposite", "mass", "keep", "climate"]].sum(), len(events))

Out of 3,106 bank-days, 243 (8%) were dropped because the same bank received opposite-sign news within 5 days, since we can’t tell which news moved the price. Of the rest, repeated campaigns inside the same 30-day run were also removed, leaving 1,478 events with non-overlapping [−10, +10] windows. 1,076 bank-days (35%) are flagged as mass events (more than 10 firms named) and 1,337 (43%) carry a climate tag. We keep both as flags, so later we can test whether the effect differs for bank-specific vs. multi-firm campaigns, and for climate vs. other campaigns, before and after COP21.

In [ ]:
# Map each event to a trading day
def calendar(file):                                                      # trading days from the Fama-French file
    d = pd.read_excel(DATA / "banks_data_bocconi" / file, usecols=["date"])["date"]
    return pd.to_datetime(d, format="%m/%d/%Y").sort_values().to_frame("trade_date")

cal = {"NA": calendar("North_America_3_Factors_Daily.xlsx"), "EU": calendar("Europe_3_Factors_Daily.xlsx")}
events["region"] = np.where(events["parent_country"] == "US", "NA", "EU")     # US banks: US calendar, others: Europe

# event day = first trading day on or after the campaign date
events = pd.concat([pd.merge_asof(g.sort_values("date"), cal[r], left_on="date", right_on="trade_date", direction="forward")
                    for r, g in events.groupby("region")]).sort_values(["ds_name", "date"]).reset_index(drop=True)
events["shifted"] = events["trade_date"] != events["date"]
print(events["shifted"].sum(), "of", len(events), "events moved to the next trading day")

**Comment:** All 3,106 bank-days already fall on trading days (Monday to Friday, none on a holiday), so no event needed to be moved. The column `trade_date` is kept as the event day for the event study.

In [ ]:
#Save the handoff files 
events.to_csv(DATA / "events_banks.csv", index=False)
banks[["ds_name", "company", "company_parent", "parent_country"]].drop_duplicates().to_csv(DATA / "lookup_banks.csv", index=False)

### **Figure 1: When, against whom, by whom**
We plot all bank-days with a campaign (before the 30-day rule) to see how the campaigns are spread over time, across banks and across NGOs.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import matplotlib.dates as mdates

# Palette: red = negative, green = positive, grey = neutral (same as the PC Labs)
C = {"neg": "#c8283c", "neu": "#b7b5ad", "pos": "#1f8a5b", "navy": "#0d3b78", "grey": "#52514e"}
events["sign"] = np.select([events["sentiment"] < 0, events["sentiment"] > 0], ["neg", "pos"], "neu")
COP21 = pd.Timestamp("2015-12-12")                         # Paris Agreement

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.3, 1, 1.1], "wspace": 0.95})

# A. campaigns per quarter by tone
q = events.groupby([events["date"].dt.to_period("Q").dt.to_timestamp(), "sign"]).size().unstack(fill_value=0)
ax = axes[0]; bottom = np.zeros(len(q))
for s, lab in [("neg", "Negative"), ("neu", "Neutral"), ("pos", "Positive")]:
    ax.bar(q.index, q[s], width=70, bottom=bottom, color=C[s], label=lab); bottom += q[s].values
ax.axvline(COP21, color="black", ls=":", lw=1); ax.text(COP21, ax.get_ylim()[1] * 0.97, " COP21", va="top", fontsize=9)
ax.set_title("A. Campaigns per quarter, by tone", loc="left", fontweight="bold")
ax.set_ylabel("Bank-days with a campaign"); ax.legend(frameon=False, loc="upper left", fontsize=9)

# B. most targeted banks
top = events.groupby("ds_name").size().nlargest(12).index
b = events[events["ds_name"].isin(top)].groupby(["ds_name", "sign"]).size().unstack(fill_value=0).reindex(top)[::-1]
ax = axes[1]; left = np.zeros(len(b))
for s in ["neg", "neu", "pos"]:
    ax.barh(b.index.str.title(), b[s], left=left, color=C[s], height=0.7); left += b[s].values
ax.set_title("B. 12 most targeted banks", loc="left", fontweight="bold"); ax.set_xlabel("Bank-days with a campaign")

# C. most active NGOs
ngo_top = events.groupby("ngo").size().nlargest(10)[::-1]
ax = axes[2]; ax.barh([x[:26] + ("…" if len(x) > 26 else "") for x in ngo_top.index], ngo_top.values, color=C["navy"], height=0.7)
ax.set_title("C. 10 most active lead NGOs", loc="left", fontweight="bold"); ax.set_xlabel("Bank-days with a campaign")

for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, axis="x" if ax is not axes[0] else "y"); ax.set_axisbelow(True)

# Panel A: one x label every 2 years (must come after the figure is created)
axes[0].xaxis.set_major_locator(mdates.YearLocator(2))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

share = (events["sign"] == "neg").mean()
fig.suptitle(f"Figure 1: Bank campaigns are mostly negative ({share:.0%}) and concentrated in a few banks and NGOs",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold")
fig.text(0.01, -0.02, f"{len(events):,} bank-days, {events['ds_name'].nunique()} banks, 2010-2018. "
         "One row per bank and day (all sentiments, before the 30-day rule). COP21 = Paris Agreement, 12 Dec 2015.", fontsize=9, color=C["grey"])
plt.show()

**Comment:**

**A.** Campaigns against banks rise from about 60 per quarter in 2010–2011 to peaks of 120–160 in 2017–2018, and red (negative) dominates in almost every quarter. The positive share is largest around COP21 (late 2015), when NGOs also praised banks for climate commitments.

**B.** A few large banks are the main targets: HSBC (about 215 bank-days), BNP Paribas (about 200) and Deutsche Bank (about 195). Deutsche Bank is almost purely negative, while BNP Paribas and ING also receive a visible share of positive campaigns.

**C.** Activity is concentrated in a handful of NGOs: Friends of the Earth France leads with about 225 bank-days, followed by Which? (UK) at about 130 and Rainforest Action Network at about 110. Most are national or regional groups (France, UK, US, Belgium, Netherlands, Spain), so a few NGOs' repeated campaigns drive much of the sample.

### Figure 2: What a campaign looks like
We describe the three Sigwatch intensity variables: where the bank is mentioned (`prominence`), the reach of the strongest NGO (`ngo_power`) and how many firms are named in the same communication (`n_targets`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"wspace": 0.3})

# A. prominence, negative vs positive campaigns (share within each tone)
ax = axes[0]; w = 0.38; lv = [1, 2, 3, 4]
for k, (s, lab) in enumerate([("neg", "Negative"), ("pos", "Positive")]):
    sh = events[events["sign"] == s]["prominence"].value_counts(normalize=True).reindex(lv, fill_value=0)
    ax.bar(np.array(lv) + (k - 0.5) * w, sh.values, width=w, color=C[s], label=lab)
ax.set_xticks(lv, ["1\nreport only", "2\nelsewhere", "3\nfirst paragraph", "4\nheadline"], fontsize=9)
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.legend(frameon=False)
ax.set_title("A. Where is the bank mentioned?", loc="left", fontweight="bold"); ax.set_ylabel("Share of campaigns")

# B. reach of the strongest NGO
ax = axes[1]; p = events["ngo_power"].value_counts(normalize=True).sort_index()
ax.bar(p.index.astype(str), p.values, color=C["navy"], width=0.7)
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.set_xlabel("ngo_power (0.5 = local, 2.75 = global)")
ax.set_title("B. Reach of the strongest NGO", loc="left", fontweight="bold")

# C. firms named in the same communication
ax = axes[2]; clip = events["n_targets"].clip(upper=30)
ax.hist(clip, bins=np.arange(0.5, 31.5, 1), color=C["navy"], weights=np.ones(len(clip)) / len(clip))
ax.axvline(10.5, color=C["neg"], ls="--", lw=1.2)
ax.text(11, ax.get_ylim()[1] * 0.92, f"{events['mass'].mean():.0%} of campaigns\nname more than 10 firms\n(flagged 'mass')", color=C["neg"], fontsize=9, va="top")
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.set_xlabel("Firms named in the same communication (30 = 30 or more)")
ax.set_title("C. Many campaigns are not bank-specific", loc="left", fontweight="bold")

for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, axis="y"); ax.set_axisbelow(True)
fig.suptitle("Figure 2: A typical campaign is mid-prominence, from a mid-reach NGO, and often names many firms",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold", y=1.03)
plt.show()

**Comment:**

**A.** Negative campaigns put the bank in more visible places than positive ones: about 24% mention it in the headline versus 14% of positive campaigns. Positive news is mostly mentioned elsewhere in the text (about 61%).

**B.** The reach of the lead NGO is bimodal: about 37% of campaigns come from NGOs with reach 1.0 (local or national groups) and 31% from NGOs at 2.5 (international groups). Very local (0.5) and very global (2.75) NGOs are rare.


**C.** Most campaigns name few firms (a peak at 5 to 6), but 35% name more than 10, and a small group names 30 or more. These are bank rankings and multi-company reports, not bank-specific news, so we flag them as `mass` and use them as a robustness check.

## **2. Market data**
We build one daily panel with returns for the 67 banks in our events, plus the Fama-French factors. US banks use the North America factors and all other banks the Europe factors. We keep only banks that appear in the campaign data.

In [ ]:
# Total-return indices of all banks (main file + Swiss/Nordic file), same bank key as the events file
def load_ri(file):
    sheets = pd.read_excel(DATA / "banks_data_bocconi" / file, sheet_name=None)
    sheets.pop("REQUEST_TABLE")
    return pd.concat([v.rename(columns={"Name": "date"}).melt(id_vars="date", var_name="bank", value_name="ri").assign(country=k)
                      for k, v in sheets.items()], ignore_index=True)

ri = pd.concat([load_ri("banks_ri.xlsm"), load_ri("scandinavian_banks_ri.xlsm")], ignore_index=True)
ri["date"] = pd.to_datetime(ri["date"], errors="coerce")
ri["ds_name"] = ri["bank"].str.rsplit(" - ", n=1).str[0].str.strip().str.lower()
ri = ri[ri["ds_name"].isin(events["ds_name"])].dropna(subset=["date", "ri"]).sort_values(["ds_name", "date"]).drop_duplicates(["ds_name", "date"])

In [ ]:
# Daily returns; drop impossible moves (> +/-100% in a day = data errors) and flag very large ones (> 50%)
ri["ret"] = ri.groupby("ds_name")["ri"].pct_change()
ri = ri.dropna(subset=["ret"])
ri["ret"] = ri["ret"].where(ri["ret"].abs() <= 1)
ri["big"] = ri["ret"].abs() > 0.5

In [ ]:
# Fama-French factors: % -> decimals, US banks use North America, all others Europe
def load_ff(file, region):
    ff = pd.read_excel(DATA / "banks_data_bocconi" / file)
    ff["date"] = pd.to_datetime(ff["date"], format="%m/%d/%Y")
    ff[["Mkt-RF", "SMB", "HML", "RF"]] /= 100
    return ff.rename(columns={"Mkt-RF": "mkt_rf", "SMB": "smb", "HML": "hml", "RF": "rf"}).assign(region=region)

ff = pd.concat([load_ff("North_America_3_Factors_Daily.xlsx", "NA"), load_ff("Europe_3_Factors_Daily.xlsx", "EU")])
ri["region"] = np.where(ri["country"] == "US", "NA", "EU")

In [ ]:
# Final panel: one row per bank and day, with excess return (return - risk-free)
panel = ri.merge(ff, on=["date", "region"], how="inner")
panel["exret"] = panel["ret"] - panel["rf"]
panel = panel[["date", "ds_name", "country", "region", "ret", "exret", "mkt_rf", "smb", "hml", "rf", "big"]].dropna(subset=["ret"])

panel.to_csv(DATA / "panel_returns.csv", index=False)                       # handoff file for C
print(f"{len(panel):,} bank-days | {panel['ds_name'].nunique()} banks | {panel['date'].min().date()} to {panel['date'].max().date()}")
print(panel.groupby("region")["ds_name"].nunique())

**Comment:** The panel has 212,542 daily observations for 72 banks (24 US, 48 European) from January 2008 to September 2020. Every bank in the campaign data has market data, so no bank is lost at this stage. A few events (about 3%) for banks with a short history will lack a full 250-day estimation window, and we handle them in the next step.

### Figures 3-4: How the bank panel behaves
We build an equal-weight portfolio of the banks in each region and compare it with the market. Figure 3 shows performance and Figure 4 shows how risky the banks are (beta) and how reliable their prices are.

In [ ]:
# Equal-weight bank portfolio, market, bank/market ratio and rolling 250-day beta, per region
plot = panel[panel["date"].between("2008-01-03", "2020-04-28")]           # 59 of 67 banks stop on 28 Apr 2020
REG = {"NA": ("US banks", C["navy"]), "EU": ("European banks", C["neg"])}
EVENTS = {"Lehman": "2008-09-15", "Euro crisis": "2011-08-01", "COP21": "2015-12-12", "Covid": "2020-03-11"}
res = {}
for r in REG:
    d = plot[plot["region"] == r]
    bank = d.groupby("date")["ret"].mean()
    mkt = d.groupby("date")[["mkt_rf", "rf"]].first().sum(axis=1)                    # market return = Mkt-RF + RF
    res[r] = pd.DataFrame({"bank": 100 * (1 + bank).cumprod(), "mkt": 100 * (1 + mkt).cumprod(),
                           "beta": bank.rolling(250).cov(mkt) / mkt.rolling(250).var()})
    res[r]["rel"] = 100 * res[r]["bank"] / res[r]["mkt"]

import matplotlib.ticker
def marks(ax):                                                            # vertical lines for the key episodes
    for lab, d in EVENTS.items():
        left = lab in ("Lehman", "Euro crisis")
        ax.axvline(pd.Timestamp(d), color="black", ls=":", lw=0.9)
        ax.text(pd.Timestamp(d), 1.0, (" " + lab) if left else (lab + " "), transform=ax.get_xaxis_transform(),
                fontsize=8.5, va="top", ha="left" if left else "right")
def style(axes):
    for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25); ax.set_axisbelow(True)

In [ ]:
# Figure 3 (performance)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.3})

# A. levels, with end values labelled
ax = axes[0]
for r, (lab, col) in REG.items():
    ax.plot(res[r].index, res[r]["bank"], color=col, lw=1.6)
    ax.plot(res[r].index, res[r]["mkt"], color=col, lw=1.1, ls="--", alpha=0.55)
    ax.annotate(f"{lab}\n{res[r]['bank'].iloc[-1]:.0f}", (res[r].index[-1], res[r]["bank"].iloc[-1]), xytext=(5, 0), textcoords="offset points", fontsize=8.5, color=col, va="center")
    ax.annotate(f"market {res[r]['mkt'].iloc[-1]:.0f}", (res[r].index[-1], res[r]["mkt"].iloc[-1]), xytext=(5, 0), textcoords="offset points", fontsize=8, color=C["grey"], va="center")
ax.set_yscale("log"); ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_yticks([30, 50, 100, 200]); ax.set_yticklabels(["30", "50", "100", "200"]); ax.set_xlim(right=pd.Timestamp("2022-01-01"))
marks(ax); ax.set_title("A. Banks (solid) vs market (dashed), 100 in Jan 2008", loc="left", fontweight="bold")

# B. bank / market ratio
ax = axes[1]
for r, (lab, col) in REG.items(): ax.plot(res[r].index, res[r]["rel"], color=col, lw=1.6, label=lab)
ax.axhline(100, color=C["grey"], lw=0.8); marks(ax)
ax.set_title("B. Banks relative to their market (100 = equal)", loc="left", fontweight="bold"); ax.legend(frameon=False, loc="lower center")

style(axes)
fig.suptitle("Figure 3: Banks fell harder than the market in crises, and European banks lag their market since 2010",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold")
n_end = (panel.groupby("ds_name")["date"].max() <= "2020-04-28").sum()
cov = plot.groupby(["date", "region"])["ds_name"].nunique().unstack()
fig.text(0.01, -0.02, f"Equal-weight portfolio of the {panel['ds_name'].nunique()} event banks; market = Mkt-RF + RF. Plot stops on 28 Apr 2020, when {n_end} of {panel['ds_name'].nunique()} banks end. "
         f"Banks with data: {cov['EU'].min()}-{cov['EU'].max()} European, {cov['NA'].min()}-{cov['NA'].max()} US.", fontsize=8.5, color=C["grey"])
plt.show()

**Comment:**

**A.** Bank stocks fall much further than the market in every crisis. In 2008-09 both bank portfolios lose about 70%, and both fall sharply again in the euro crisis (2011-12) and in the Covid crash (March 2020). Between the crises the two regions diverge. US banks stay above their market for most of 2013-2019, but after the Covid crash they stand at 199 against 240 for the market on 28 April 2020. European banks move around 100 for twelve years while their market rises to about 140 before Covid, and end at 66 against 107. The COP21 line falls in a flat period for both, with no visible reaction at aggregate level.

**B.** The ratio makes the gap clear. US banks move between about 85 and 110 until 2016, jump to about 125 after the US election at the end of 2016 (which matters for the Trump effect we test later), peak near 130 in 2018 and fall to about 75 in March 2020. European banks rise to about 125 in 2010, fall to about 78 in 2012, recover to about 100 in 2015, and then drift down to about 77 before Covid and 60 in April 2020. The underperformance of European banks is persistent, not a one-off shock.

In [ ]:
#Figure 4 (risk and data quality)
# Share of days with exactly zero return per bank (stale prices), from 2010
zero = (panel[panel["date"] >= "2010-01-01"].assign(z=lambda d: d["ret"].abs() < 1e-12)
        .groupby("ds_name")["z"].mean().sort_values(ascending=False))
reg_of = panel.groupby("ds_name")["region"].first()
top = zero.head(12)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.55, "width_ratios": [1.2, 1]})

# A. rolling beta of the bank portfolio
ax = axes[0]
for r, (lab, col) in REG.items(): ax.plot(res[r].index, res[r]["beta"], color=col, lw=1.6, label=lab)
ax.axhline(1, color=C["grey"], lw=0.8); marks(ax)
ax.set_title("A. Bank portfolio beta (rolling 250 days)", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.55, 0.92))

# B. banks with the most stale prices
ax = axes[1]
ax.barh([s[:26] for s in top.index], top.values * 100, color=[REG[reg_of[s]][1] for s in top.index], height=0.65)
ax.set_title("B. Banks with the most zero-return days (2010-20)", loc="left", fontweight="bold")
ax.set_xlabel("% of trading days with zero return")

style(axes)
fig.suptitle("Figure 4: Bank betas move a lot over time, and a few banks have stale prices", x=0.01, ha="left", fontsize=13.5, fontweight="bold")
fig.text(0.01, -0.02, "Beta of the equal-weight bank portfolio on the market. Bars: red = European, blue = US bank. A zero return means the price did not change that day.",
         fontsize=8.5, color=C["grey"])
plt.show()

**Comment:**

**A.** The beta of the bank portfolio is far from constant. For US banks it ranges from about 0.9 to about 2.4: it peaks at about 2.4 in 2010 after the financial crisis, falls to about 1.2 in 2011, rises again to about 1.6 in 2017, and falls back to about 0.9 in 2019 before rising with Covid. European banks are lower, mostly between 0.75 and 1.25, with a minimum of about 0.65 in 2018. A beta estimated in one period is therefore a poor guide to another, so each event must get its own beta from a window just before it (the 250 days ending 50 days before the event), not one beta for the whole sample.

**B.** A few banks have prices that do not change on many days. Monte dei Paschi has a zero return on about 34% of days and the Swiss National Bank on about 23%. Most banks are far lower (the median is about 4-5%, and only these two are above 20%). Eleven of the twelve banks shown are European (the exception is Ally Financial). For these banks the beta is estimated with more noise, and an event day with no price change can hide a real reaction. We should check that these banks do not drive the results. We also dropped Saga Group, whose Datastream index never changes (100% zero returns), because it has no usable price data, and removed a duplicated Crédit Agricole series that created false zero returns.

In [ ]:
panel = panel.sort_values(["ds_name", "date"]).reset_index(drop=True)
panel["t"] = panel.groupby("ds_name").cumcount()
print(panel.groupby("ds_name")["t"].agg(["min", "max"]).describe().round(0))

In [ ]:
ev = events[events["keep"]].copy()

ev = ev.merge(
    panel[["ds_name", "date", "t"]], 
    left_on=["ds_name", "trade_date"],
    right_on=["ds_name", "date"],
    how="left",  
    suffixes=("", "_panel"),
)
ev = ev.rename(columns={"t": "t0"})

In [ ]:
print(f"{len(ev):,} events kept")
print(f"{ev['t0'].isna().sum()} without a matching trading day in the panel")
print(f"{(ev['t0'] < 300).sum()} with less than 300 days of history before t0")

In [ ]:
miss = ev[ev["t0"].isna()]
span = panel.groupby("ds_name")["date"].agg(first="min", last="max")
miss = miss.join(span, on="ds_name")

outside = (miss["trade_date"] < miss["first"]) | (miss["trade_date"] > miss["last"])
print(f"{outside.sum()} outside the bank's price history (not yet listed / delisted)")
print(f"{(~outside).sum()} inside the history but no return that day (holiday / removed return)")
print(f"{miss['ds_name'].nunique()} distinct banks concerned")

Locating the event day. We number each bank’s trading days (t) so that all windows are measured in trading days, and find the event day t0 by matching each kept event to the panel on bank and trade date. Of the 1,478 kept events, 57 (8 banks) fall outside the bank’s price history: the bank was not yet listed, or no longer listed, on the event date, so no market reaction can be measured. No event is lost to holidays or removed returns. A further 14 events have fewer than 300 trading days before t0, so the estimation window [t0−300, t0−50] cannot be filled. Dropping both leaves 1,407 events (95% of the kept sample).

In [ ]:
ev = ev[ev["t0"].notna() & (ev["t0"] >= 300)].copy()
ev["t0"] = ev["t0"].astype(int)          # t0 était en float à cause des NaN
print(f"{len(ev):,} events for the event study")

In [ ]:
e = ev.iloc[0]

est = panel[
    (panel["ds_name"] == e["ds_name"])
    & (panel["t"] >= e["t0"] - 300)
    & (panel["t"] <= e["t0"] - 50)
]
print(len(est), "days in the estimation window")

win = panel[
    (panel["ds_name"] == e["ds_name"])
    & (panel["t"] >= e["t0"] - 10)
    & (panel["t"] <= e["t0"] + 10)
]
print(len(win), "days in the event window")

In [ ]:
x, y = est["mkt_rf"], est["exret"]

beta = x.cov(y) / x.var()
alpha = y.mean() - beta * x.mean()
print(f"alpha = {alpha:.5f}, beta = {beta:.2f}")

In [ ]:
win = win.copy()

win["ar"] = win["exret"] - (alpha + beta * win["mkt_rf"])
win["car"] = win["ar"].cumsum()
print(f"{e['ds_name']}: CAR[-10,+10] = {win['car'].iloc[-1]:.2%}, "
      f"AR on event day = {win.loc[win['t'] == e['t0'], 'ar'].iloc[0]:.2%}")

In [ ]:
rows = []


for _, e in ev.iterrows():
    bank = panel[panel["ds_name"] == e["ds_name"]]

    est = bank[(bank["t"] >= e["t0"] - 300) & (bank["t"] <= e["t0"] - 50)]
    win = bank[(bank["t"] >= e["t0"] - 10) & (bank["t"] <= e["t0"] + 10)].copy()

    if len(est) < 200 or len(win) < 21:
        continue

    x, y = est["mkt_rf"], est["exret"]

    beta = x.cov(y) / x.var()
    alpha = y.mean() - beta * x.mean()
    
    win["ar"] = win["exret"] - (alpha + beta * win["mkt_rf"])
    win["car"] = win["ar"].cumsum()

    rows.append({
        "ds_name": e["ds_name"],
        "trade_date": e["trade_date"],
        "alpha": alpha,
        "beta": beta,
        "car": win["car"].iloc[-1],
        "ar0": win["ar"].iloc[10],
    })
    

cars = pd.DataFrame(rows)

In [ ]:
print(f"{len(cars):,} events with a CAR")
print(cars[["beta", "car", "ar0"]].describe().round(4))

In [ ]:
print(f"{(cars['car'] < -0.5).sum()} events with CAR < -50%, {(cars['car'] > 0.3).sum()} with CAR > +30%")
print(cars.nsmallest(5, "car")[["ds_name", "trade_date", "car", "beta"]].round(3))
print(cars.nlargest(3, "car")[["ds_name", "trade_date", "car", "beta"]].round(3))